# Predicția prețului unei mașini second-hand

**Autor:** Andrei Motau  
**Tip problemă:** regresie  
**Variabilă țintă:** `priceUSD`

Notebook-ul prezintă analiza exploratorie, deciziile de curățare, ingineria caracteristicilor, compararea modelelor și interpretarea modelului final.

In [ ]:
from pathlib import Path
import json
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split

ROOT = Path.cwd()
if not (ROOT / 'data' / 'cars.csv').exists():
    ROOT = ROOT.parent

from src.data_cleaning import clean_data
from src.feature_engineering import add_features
from src.data_preprocessing import split_features_target
from src.model_evaluation import regression_metrics, prediction_examples

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', 30)
DATA_PATH = ROOT / 'data' / 'cars.csv'
MODEL_PATH = ROOT / 'models' / 'car_price_model.joblib'
REPORTS_PATH = ROOT / 'reports'
ROOT

## 1. Încărcarea și înțelegerea datelor

In [ ]:
raw = pd.read_csv(DATA_PATH)
print(f'Rânduri: {raw.shape[0]:,} | Coloane: {raw.shape[1]}')
display(raw.head())
display(raw.dtypes.rename('tip_date').to_frame())

In [ ]:
quality = pd.DataFrame({
    'valori_lipsă': raw.isna().sum(),
    'procent_lipsă': (raw.isna().mean() * 100).round(2),
    'valori_unice': raw.nunique(dropna=False),
})
print(f'Duplicate complete: {raw.duplicated().sum():,}')
display(quality)

Observații principale:

- există valori lipsă la `volume(cm3)`, `drive_unit` și `segment`;
- există duplicate complete;
- ținta este puternic asimetrică spre dreapta;
- câteva valori pentru kilometraj și cilindree sunt improbabile și necesită verificare.

In [ ]:
display(raw[['priceUSD', 'year', 'mileage(kilometers)', 'volume(cm3)']].describe(
    percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
).T)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(raw['priceUSD'], bins=60, ax=axes[0])
axes[0].set_title('Distribuția prețului')
axes[0].set_xlabel('Preț (USD)')
sns.histplot(np.log1p(raw['priceUSD']), bins=60, ax=axes[1], color='darkorange')
axes[1].set_title('Distribuția log(1 + preț)')
axes[1].set_xlabel('log(1 + priceUSD)')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
raw['make'].value_counts().head(15).sort_values().plot.barh(ax=axes[0], color='steelblue')
axes[0].set_title('Top 15 mărci după numărul de anunțuri')
axes[0].set_xlabel('Anunțuri')
sns.scatterplot(
    data=raw.sample(min(5000, len(raw)), random_state=42),
    x='year', y='priceUSD', alpha=0.35, ax=axes[1]
)
axes[1].set_title('Preț în funcție de anul fabricației')
axes[1].set_ylim(0, raw['priceUSD'].quantile(0.99))
plt.tight_layout()
plt.show()

## 2. Curățarea datelor

In [ ]:
cleaned = clean_data(raw)
print(f'Înainte: {len(raw):,} rânduri')
print(f'După curățare: {len(cleaned):,} rânduri')
print(f'Rânduri eliminate: {len(raw) - len(cleaned):,}')
display(cleaned.head())
display(cleaned.isna().sum().rename('valori_lipsă').to_frame())

Curățarea elimină duplicatele și doar rândurile imposibile sau foarte improbabile. Valorile lipsă ale predictorilor nu sunt șterse: ele sunt imputate în pipeline, astfel încât pierdem cât mai puține anunțuri.

## 3. Ingineria caracteristicilor

In [ ]:
featured = add_features(cleaned)
new_features = ['car_age', 'mileage_per_year', 'log_mileage',
                'engine_volume_liters', 'make_model']
display(featured[new_features].head())
display(featured[new_features[:-1]].describe().T)

Caracteristicile noi au roluri clare:

- `car_age`: deprecierea este legată mai direct de vechime decât de anul brut;
- `mileage_per_year`: diferențiază utilizarea intensă de kilometrajul acumulat natural;
- `log_mileage`: reduce influența valorilor extreme;
- `engine_volume_liters`: cilindree într-o scară interpretabilă;
- `make_model`: surprinde combinația marcă-model.

## 4. Preprocesare și antrenare

Pipeline-ul modular din `src/` aplică imputare mediană și standardizare variabilelor numerice, respectiv imputarea modei și One-Hot Encoding variabilelor categorice. Ținta este transformată cu `log1p` pentru a reduce efectul prețurilor foarte mari.

Toate modelele folosesc aceeași împărțire 80/20 și `random_state=42`. Pentru refacerea completă a antrenării se rulează:

```bash
python -m src.model_training --data data/cars.csv --output-dir .
```

## 5. Compararea modelelor

In [ ]:
comparison = pd.read_csv(REPORTS_PATH / 'model_comparison.csv')
comparison_display = comparison.copy()
comparison_display[['MAE', 'MSE', 'RMSE', 'R2']] = comparison_display[
    ['MAE', 'MSE', 'RMSE', 'R2']
].round(2)
display(comparison_display)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.barplot(data=comparison, x='MAE', y='model', ax=axes[0], color='steelblue')
axes[0].set_title('MAE mai mic este mai bun')
axes[0].set_xlabel('MAE (USD)')
sns.barplot(data=comparison, x='R2', y='model', ax=axes[1], color='seagreen')
axes[1].set_title('R² mai mare este mai bun')
axes[1].set_xlim(0.75, 1.0)
plt.tight_layout()
plt.show()

## 6. Evaluarea modelului final

In [ ]:
X, y = split_features_target(featured)
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
final_model = joblib.load(MODEL_PATH)
predictions = np.maximum(final_model.predict(X_test), 0)
metrics = regression_metrics(y_test, predictions)
display(pd.Series(metrics, name='valoare').round(4).to_frame())
display(prediction_examples(y_test, predictions, count=12))

In [ ]:
plot_limit = np.quantile(y_test, 0.99)
mask = y_test.to_numpy() <= plot_limit
plt.figure(figsize=(7, 6))
plt.scatter(y_test.to_numpy()[mask], predictions[mask], alpha=0.25, s=12)
plt.plot([0, plot_limit], [0, plot_limit], '--', color='crimson', label='predicție perfectă')
plt.xlabel('Preț real (USD)')
plt.ylabel('Preț estimat (USD)')
plt.title('Preț real comparat cu prețul estimat')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Exemplul cerut

In [ ]:
example = pd.DataFrame([{
    'make': 'volkswagen', 'model': 'golf', 'priceUSD': 1,
    'year': 2014, 'condition': 'with mileage',
    'mileage(kilometers)': 180000, 'fuel_type': 'diesel',
    'volume(cm3)': 1600, 'color': 'black', 'transmission': 'mechanics',
    'drive_unit': 'front-wheel drive', 'segment': 'C'
}])
example_prepared = add_features(clean_data(example))
example_X, _ = split_features_target(example_prepared)
estimated_price = max(float(final_model.predict(example_X)[0]), 0)
print(f'Preț estimat pentru Volkswagen Golf: ${estimated_price:,.2f}')

## Concluzie

Modelul final este ales după cel mai mic MAE. MAE este criteriul principal deoarece arată direct cu câți dolari greșește modelul în medie. R² arată proporția din variația prețului explicată de model, iar RMSE penalizează mai puternic erorile mari.

Rezultatele indică faptul că modelele de tip ensemble surprind mai bine relațiile neliniare dintre marcă, model, vechime, kilometraj și preț decât regresia liniară. Predicția rămâne orientativă: istoricul de service, dotările și starea reală nu sunt prezente în date.